# Chapter 4: Design a Rate Limiter
*System Design Interview*

## TL;DR

A rate limiter controls traffic by limiting the number of requests a client can send in a time window. Key design decisions:

| Decision | Options |
|----------|---------|
| **Placement** | Client-side, server-side, middleware/API gateway |
| **Algorithm** | Token bucket, leaking bucket, fixed window, sliding window log, sliding window counter |
| **Storage** | Redis (in-memory, fast, supports INCR + EXPIRE) |
| **Distributed** | Centralized Redis, Lua scripts for atomicity |

**Benefits:** DoS protection, cost reduction, server overload prevention

## Requirements

**Functional:**
- Accurately limit excessive requests based on configurable rules
- Return HTTP 429 (Too Many Requests) with rate-limit headers
- Support different throttle rules (per user, per IP, per API endpoint)

**Non-Functional:**
- Low latency (must not slow down HTTP response time)
- Memory efficient
- Distributed -- shared across multiple servers/processes
- High fault tolerance -- failure must not bring down the system
- Exception handling -- clear feedback to throttled users

## Estimation: Rate Limiter for a Large API

In [ ]:
# Rate limiter capacity estimation
print("=== Rate Limiter Capacity Estimation ===")
print()

# Assumptions
total_api_qps = 500_000  # 500K requests/second across all endpoints
num_unique_users = 10_000_000  # 10M unique users/day
rules_per_user = 3  # e.g., POST limit, GET limit, global limit
counter_size_bytes = 8 + 8 + 8  # user_id (8) + counter (8) + timestamp (8) = 24 bytes

print(f"Total API QPS:           {total_api_qps:>12,}")
print(f"Unique users/day:        {num_unique_users:>12,}")
print(f"Rules per user:          {rules_per_user:>12}")
print(f"Counter size:            {counter_size_bytes:>12} bytes")
print()

# Memory for counters
total_counters = num_unique_users * rules_per_user
memory_bytes = total_counters * counter_size_bytes
memory_mb = memory_bytes / (1024**2)
memory_gb = memory_bytes / (1024**3)

print(f"Total counters:          {total_counters:>12,}")
print(f"Memory needed:           {memory_mb:>12,.0f} MB ({memory_gb:.2f} GB)")
print()

# Redis can handle this easily
print("Redis single node: ~25 GB memory, 100K+ ops/sec")
print(f"We need ~{memory_gb:.1f} GB -> fits in a single Redis node")
print(f"At {total_api_qps:,} QPS, we may need Redis cluster or read replicas")

## High-Level Architecture

```mermaid
graph TB
    CLIENT[Client] --> MW[Rate Limiter Middleware]
    MW --> RULES_CACHE[(Rules Cache)]
    MW --> REDIS[(Redis - Counters)]

    RULES_WORKER[Rules Worker] -->|Pull rules| DISK[(Rules on Disk)]
    RULES_WORKER -->|Update| RULES_CACHE

    MW -->|Allowed| API[API Servers]
    MW -->|Rejected| R429[HTTP 429 + Headers]

    subgraph Rate Limit Headers
        H1[X-Ratelimit-Remaining]
        H2[X-Ratelimit-Limit]
        H3[X-Ratelimit-Retry-After]
    end
```

## Rate Limiting Algorithms Compared

### 1. Token Bucket
```mermaid
graph LR
    REFILL[Refill: N tokens/sec] --> BUCKET[Token Bucket - capacity C]
    BUCKET -->|Has token| ALLOW[Request Allowed]
    BUCKET -->|Empty| DENY[Request Denied]
    REQ[Request] --> BUCKET
```
- **Used by:** Amazon, Stripe
- **Params:** bucket size, refill rate
- **Allows bursts** up to bucket size

### 2. Leaking Bucket
```mermaid
graph LR
    REQ[Requests] --> Q[FIFO Queue - size N]
    Q -->|Fixed rate| PROC[Process Request]
    Q -->|Queue full| DROP[Drop Request]
```
- **Used by:** Shopify
- **Params:** queue size, outflow rate
- **Smooth output** at fixed rate

### 3. Fixed Window Counter
- Divide time into fixed windows (e.g., 1-second intervals)
- Count requests per window; reject when count exceeds threshold
- **Problem:** Burst at window edges can allow 2x the limit

### 4. Sliding Window Log
- Track timestamps of all requests in a sorted set
- Remove outdated entries; count remaining
- **Accurate** but **memory-heavy** (stores every timestamp)

### 5. Sliding Window Counter
- Hybrid: weighted average of current + previous window
- Formula: `current_count + prev_count * overlap_pct`
- **Memory efficient**, ~0.003% error rate (Cloudflare)

## Algorithm Comparison Table

| Algorithm | Memory | Accuracy | Burst Handling | Complexity |
|-----------|--------|----------|----------------|------------|
| **Token Bucket** | Low | Good | Allows bursts | Low |
| **Leaking Bucket** | Low | Good | Smooths output | Low |
| **Fixed Window** | Low | Edge spikes (2x) | Poor at edges | Very Low |
| **Sliding Window Log** | High | Exact | Perfect | Medium |
| **Sliding Window Counter** | Low | ~99.997% | Good (smoothed) | Medium |

In [ ]:
# Simulate Token Bucket Algorithm
class TokenBucket:
    def __init__(self, capacity, refill_rate):
        self.capacity = capacity
        self.tokens = capacity
        self.refill_rate = refill_rate  # tokens per second
        self.last_refill = 0

    def allow_request(self, current_time):
        # Refill tokens
        elapsed = current_time - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_rate)
        self.last_refill = current_time

        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

# Simulate: 4 tokens capacity, 2 tokens/sec refill
bucket = TokenBucket(capacity=4, refill_rate=2)

# Simulate requests at various times
requests = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0]

print(f"{'Time':>6}  {'Tokens Before':>14}  {'Allowed':>8}  {'Tokens After':>13}")
print("-" * 48)
for t in requests:
    tokens_before = min(bucket.capacity, bucket.tokens + (t - bucket.last_refill) * bucket.refill_rate)
    allowed = bucket.allow_request(t)
    print(f"{t:>6.1f}  {tokens_before:>14.1f}  {'YES' if allowed else 'NO':>8}  {bucket.tokens:>13.1f}")

In [ ]:
# Simulate Sliding Window Counter
def sliding_window_counter(prev_count, curr_count, overlap_pct, limit):
    """
    Calculate effective request count using sliding window counter.
    prev_count: requests in previous window
    curr_count: requests in current window
    overlap_pct: how much of the previous window overlaps with the sliding window
    limit: max allowed requests
    """
    estimated = curr_count + prev_count * overlap_pct
    return estimated, estimated <= limit

# Example from the book: max 7 requests/min
# Previous window: 5 requests, Current window: 3 requests
# New request arrives at 30% into current window -> 70% overlap with prev
print("=== Sliding Window Counter Example ===")
print(f"Limit: 7 requests per minute")
print(f"Previous window: 5 requests")
print(f"Current window: 3 requests")
print()

for position_pct in [0.3, 0.5, 0.7, 0.9]:
    overlap = 1 - position_pct
    estimated, allowed = sliding_window_counter(5, 3, overlap, 7)
    status = "ALLOWED" if allowed else "REJECTED"
    print(f"At {position_pct*100:.0f}% into window: estimated = 3 + 5 * {overlap:.1f} = {estimated:.1f}  -> {status}")

## Deep Dive: Distributed Challenges

### Race Condition Problem

```mermaid
sequenceDiagram
    participant R1 as Request 1
    participant R2 as Request 2
    participant Redis

    R1->>Redis: READ counter (= 3)
    R2->>Redis: READ counter (= 3)
    R1->>Redis: WRITE counter = 4
    R2->>Redis: WRITE counter = 4
    Note over Redis: Bug! Should be 5, but both read 3
```

**Solutions:**
- **Lua script:** Atomic read-check-increment in Redis
- **Redis sorted sets:** Atomic operations on sorted set data structure

### Synchronization Problem

```mermaid
graph TB
    C1[Client 1] --> RL1[Rate Limiter 1]
    C1 --> RL2[Rate Limiter 2]
    C2[Client 2] --> RL1
    C2 --> RL2

    RL1 --> REDIS[(Centralized Redis)]
    RL2 --> REDIS

    style REDIS fill:#ff9999
```

**Solution:** Use centralized Redis instead of local counters. All rate limiter instances read/write from the same Redis cluster.

## Trade-offs

| Decision | Option A | Option B | Recommendation |
|----------|----------|----------|----------------|
| **Placement** | Server-side | API Gateway middleware | Gateway if using microservices |
| **Algorithm** | Token Bucket | Sliding Window Counter | Token Bucket for simplicity; SWC for accuracy |
| **Hard vs Soft limit** | Strict enforcement | Allow temporary bursts | Depends on use case |
| **Rejected requests** | Drop immediately | Queue for later | Queue for order-like systems |
| **Distributed sync** | Sticky sessions | Centralized Redis | Redis (scalable, no stickiness) |

## Rate Limiter Rules (Lyft Example)

```yaml
# Rule 1: Marketing messages
domain: messaging
descriptors:
  - key: message_type
    value: marketing
    rate_limit:
      unit: day
      requests_per_unit: 5

# Rule 2: Login attempts
domain: auth
descriptors:
  - key: auth_type
    value: login
    rate_limit:
      unit: minute
      requests_per_unit: 5
```

Rules are stored on disk, pulled by workers, and cached in memory for fast access.

## Monitoring and Tuning

After deployment, continuously monitor:

| Metric | Action if Off |
|--------|--------------|
| Drop rate too high | Rules too strict -- relax thresholds |
| Drop rate too low | Rules too lenient -- tighten |
| Burst traffic (flash sales) | Switch to token bucket algorithm |
| Latency increase | Add more Redis replicas or edge servers |
| Memory usage | Optimize counter storage or add sharding |

## Takeaways

1. **Rate limiters protect systems** from DoS, reduce cost, and prevent overload
2. **Token bucket** is the most common algorithm (Amazon, Stripe) -- simple and allows bursts
3. **Sliding window counter** offers the best accuracy-to-memory ratio (~0.003% error)
4. **Redis** is the standard backing store -- fast, supports atomic operations
5. **Distributed challenges** (race conditions, sync) are solved with Lua scripts and centralized Redis
6. **Monitor and tune** rules and algorithms post-deployment

## See Also

- [[rate-limiter-placement]]
- [[rate-limiting-algorithms]]
- [[token-bucket]]
- [[sliding-window-counter]]
- [[distributed-rate-limiting]]
- [[rate-limiter-monitoring]]